error frequnecy vs catastrophic failures


In [0]:
%sql
WITH error_failure AS (
    SELECT
        e.machineID,
        e.errorID,
        e.datetime AS error_time,
        f.datetime AS failure_time,
        f.failure
    FROM tseries.tseries.silver_errors e
    INNER JOIN tseries.tseries.silver_failures f
        ON e.machineID = f.machineID
       AND f.datetime >= e.datetime
       AND f.datetime <= e.datetime + INTERVAL 48 HOURS
),

summary AS (
    SELECT
        errorID,

        COUNT(*) AS error_failure_events,

        COUNT(
            CASE
                WHEN failure_time <= error_time + INTERVAL 24 HOURS
                THEN 1
            END
        ) AS failures_within_24h,

        COUNT(
            CASE
                WHEN failure_time > error_time + INTERVAL 24 HOURS
                THEN 1
            END
        ) AS failures_24_to_48h

    FROM error_failure
    GROUP BY errorID
)

SELECT *
FROM summary
ORDER BY failures_within_24h DESC;

errorID,error_failure_events,failures_within_24h,failures_24_to_48h
error3,318,300,18
error2,309,298,11
error1,237,218,19
error5,208,204,4
error4,169,161,8


machine age/model vs baseline telemetry

In [0]:
%sql
SELECT
    m.model,
    CASE
        WHEN m.age <= 2 THEN '0-2 years'
        WHEN m.age <= 5 THEN '3-5 years'
        WHEN m.age <= 10 THEN '6-10 years'
        ELSE '10+ years'
    END AS age_group,

    COUNT(DISTINCT m.machineID) AS machine_count,

    ROUND(AVG(t.volt), 2) AS avg_voltage,
    ROUND(AVG(t.rotate), 2) AS avg_rotation,
    ROUND(AVG(t.pressure), 2) AS avg_pressure,
    ROUND(AVG(t.vibration), 2) AS avg_vibration

FROM tseries.tseries.silver_machines m
INNER JOIN tseries.tseries.silver_telemetry t
    ON m.machineID = t.machineID

GROUP BY
    m.model,
    CASE
        WHEN m.age <= 2 THEN '0-2 years'
        WHEN m.age <= 5 THEN '3-5 years'
        WHEN m.age <= 10 THEN '6-10 years'
        ELSE '10+ years'
    END

ORDER BY
    m.model,
    age_group;

model,age_group,machine_count,avg_voltage,avg_rotation,avg_pressure,avg_vibration
model1,0-2 years,1,170.61,447.04,101.45,40.27
model1,10+ years,11,170.77,446.28,101.24,40.5
model1,3-5 years,2,170.8,446.04,101.28,40.31
model1,6-10 years,2,170.9,446.87,101.31,40.31
model2,0-2 years,1,170.99,445.34,101.32,40.28
model2,10+ years,10,170.72,446.71,101.27,40.49
model2,3-5 years,1,170.79,447.63,101.04,40.21
model2,6-10 years,5,170.65,447.13,101.13,40.25
model3,0-2 years,3,170.84,446.54,100.78,40.26
model3,10+ years,20,170.8,446.55,100.66,40.47


time since maintenance vs snsor stability

In [0]:
%sql
WITH maintenance_before_sensor AS (

    SELECT
        t.machineID,
        t.datetime AS telemetry_time,
        t.volt,
        t.rotate,
        t.pressure,
        t.vibration,

        MAX(m.datetime) AS last_maintenance_time

    FROM tseries.tseries.silver_telemetry t

    LEFT JOIN tseries.tseries.silver_maint m
        ON t.machineID = m.machineID
       AND m.datetime <= t.datetime

    GROUP BY
        t.machineID,
        t.datetime,
        t.volt,
        t.rotate,
        t.pressure,
        t.vibration
),

maintenance_age AS (

    SELECT
        *,
        DATEDIFF(
            telemetry_time,
            last_maintenance_time
        ) AS days_since_maintenance

    FROM maintenance_before_sensor
    WHERE last_maintenance_time IS NOT NULL
)

SELECT
    CASE
        WHEN days_since_maintenance <= 7 THEN '0-7 days'
        WHEN days_since_maintenance <= 30 THEN '8-30 days'
        WHEN days_since_maintenance <= 60 THEN '31-60 days'
        ELSE '60+ days'
    END AS maintenance_age_group,

    COUNT(*) AS telemetry_records,

    ROUND(AVG(volt), 2) AS avg_voltage,
    ROUND(STDDEV(volt), 2) AS voltage_stddev,

    ROUND(AVG(rotate), 2) AS avg_rotation,
    ROUND(STDDEV(rotate), 2) AS rotation_stddev,

    ROUND(AVG(pressure), 2) AS avg_pressure,
    ROUND(STDDEV(pressure), 2) AS pressure_stddev,

    ROUND(AVG(vibration), 2) AS avg_vibration,
    ROUND(STDDEV(vibration), 2) AS vibration_stddev

FROM maintenance_age

GROUP BY
    CASE
        WHEN days_since_maintenance <= 7 THEN '0-7 days'
        WHEN days_since_maintenance <= 30 THEN '8-30 days'
        WHEN days_since_maintenance <= 60 THEN '31-60 days'
        ELSE '60+ days'
    END

ORDER BY maintenance_age_group;

maintenance_age_group,telemetry_records,avg_voltage,voltage_stddev,avg_rotation,rotation_stddev,avg_pressure,pressure_stddev,avg_vibration,vibration_stddev
0-7 days,397005,170.6,15.4,447.78,51.74,100.65,10.82,40.31,5.29
31-60 days,21901,170.95,15.59,446.85,52.21,101.07,11.38,40.36,5.37
60+ days,13386,170.96,15.56,445.62,53.95,100.85,11.22,40.25,5.24
8-30 days,443808,170.92,15.6,445.57,53.46,101.03,11.23,40.46,5.44


forecast telemetry 12 hours

In [0]:
%sql
WITH recent AS (

    SELECT
        machineID,

        MAX(datetime) AS latest_time,

        AVG(
            CASE
                WHEN datetime >=
                    (SELECT MAX(datetime) FROM tseries.tseries.silver_telemetry)
                    - INTERVAL 12 HOURS
                THEN volt
            END
        ) AS avg_voltage_12h,

        AVG(
            CASE
                WHEN datetime >=
                    (SELECT MAX(datetime) FROM tseries.tseries.silver_telemetry)
                    - INTERVAL 12 HOURS
                THEN vibration
            END
        ) AS avg_vibration_12h,

        AVG(
            CASE
                WHEN datetime >=
                    (SELECT MAX(datetime) FROM tseries.tseries.silver_telemetry)
                    - INTERVAL 12 HOURS
                THEN pressure
            END
        ) AS avg_pressure_12h,

        AVG(
            CASE
                WHEN datetime >=
                    (SELECT MAX(datetime) FROM tseries.tseries.silver_telemetry)
                    - INTERVAL 12 HOURS
                THEN rotate
            END
        ) AS avg_rotation_12h

    FROM tseries.tseries.silver_telemetry

    GROUP BY machineID
)

SELECT
    machineID,
    latest_time,

    ROUND(avg_voltage_12h, 2) AS forecast_voltage_12h,
    ROUND(avg_vibration_12h, 2) AS forecast_vibration_12h,
    ROUND(avg_pressure_12h, 2) AS forecast_pressure_12h,
    ROUND(avg_rotation_12h, 2) AS forecast_rotation_12h

FROM recent
ORDER BY machineID;

machineID,latest_time,forecast_voltage_12h,forecast_vibration_12h,forecast_pressure_12h,forecast_rotation_12h
1,2016-01-01T06:00:00.000Z,169.77,40.48,96.82,436.73
2,2016-01-01T06:00:00.000Z,175.65,39.19,98.05,432.25
3,2016-01-01T06:00:00.000Z,168.57,38.1,100.19,445.19
4,2016-01-01T06:00:00.000Z,165.81,38.7,99.5,446.34
5,2016-01-01T06:00:00.000Z,162.75,40.43,151.82,452.49
6,2016-01-01T06:00:00.000Z,173.49,40.65,92.94,447.13
7,2016-01-01T06:00:00.000Z,164.8,39.07,98.34,445.2
8,2016-01-01T06:00:00.000Z,173.24,40.71,101.27,452.19
9,2016-01-01T06:00:00.000Z,169.09,39.19,100.41,459.61
10,2016-01-01T06:00:00.000Z,162.72,39.75,100.28,429.47


multi sensor anomaly

In [0]:
%sql
WITH stats AS (

    SELECT
        AVG(volt) AS avg_volt,
        STDDEV(volt) AS std_volt,

        AVG(rotate) AS avg_rotate,
        STDDEV(rotate) AS std_rotate,

        AVG(pressure) AS avg_pressure,
        STDDEV(pressure) AS std_pressure,

        AVG(vibration) AS avg_vibration,
        STDDEV(vibration) AS std_vibration

    FROM tseries.tseries.silver_telemetry
),

scored AS (

    SELECT
        t.machineID,
        t.datetime,

        t.volt,
        t.rotate,
        t.pressure,
        t.vibration,

        ABS((t.volt - s.avg_volt) / NULLIF(s.std_volt, 0))
            AS voltage_z,

        ABS((t.rotate - s.avg_rotate) / NULLIF(s.std_rotate, 0))
            AS rotation_z,

        ABS((t.pressure - s.avg_pressure) / NULLIF(s.std_pressure, 0))
            AS pressure_z,

        ABS((t.vibration - s.avg_vibration) / NULLIF(s.std_vibration, 0))
            AS vibration_z

    FROM tseries.tseries.silver_telemetry t
    CROSS JOIN stats s
)

SELECT
    *,
    ROUND(
        voltage_z +
        rotation_z +
        pressure_z +
        vibration_z,
        2
    ) AS anomaly_score,

    CASE
        WHEN
            voltage_z > 3
            OR rotation_z > 3
            OR pressure_z > 3
            OR vibration_z > 3
        THEN 'ANOMALY'
        ELSE 'NORMAL'
    END AS anomaly_status

FROM scored

ORDER BY anomaly_score DESC;

machineID,datetime,volt,rotate,pressure,vibration,voltage_z,rotation_z,pressure_z,vibration_z,anomaly_score,anomaly_status
42,2015-04-04T16:00:00.000Z,159.538475448858,618.194981417727,167.761892344279,32.3445491506669,0.7246874834748739,3.25758882504676,6.055314121769543,1.4971913930309984,11.53,ANOMALY
51,2015-01-01T09:00:00.000Z,200.612935082807,323.752111407676,144.657425486321,57.7975856545907,1.9237203551061877,2.3323323326758225,3.96416222597519,3.242348198660665,11.46,ANOMALY
56,2015-09-24T22:00:00.000Z,153.207688443226,211.202724817102,145.44860572637,32.6795794232416,1.1328853290344785,4.469053107642701,4.035770805782072,1.434806331781499,11.07,ANOMALY
84,2015-05-15T05:00:00.000Z,210.452458768933,512.621163671999,161.638446829749,31.2886730202786,2.558155278668802,1.253297406822751,5.5010899202233245,1.6938031695129936,11.01,ANOMALY
5,2015-12-31T16:00:00.000Z,197.35413342618,465.821561563448,182.111769721569,32.3808265287065,1.7135986413038105,0.3648191563877583,7.354100779190324,1.4904362827134163,10.92,ANOMALY
81,2015-06-15T10:00:00.000Z,180.601894692374,252.170576117246,145.037729261507,26.6014862050699,0.6334441907818877,3.6912891265984658,3.998582970905041,2.566591274025088,10.89,ANOMALY
74,2015-08-03T14:00:00.000Z,201.639926814057,340.700739242529,141.671712436629,57.5345362799721,1.9899389487393058,2.010567049905768,3.6939296539558617,3.1933665015109995,10.89,ANOMALY
16,2015-05-15T12:00:00.000Z,146.058172998836,513.91862560257,144.763344463502,61.5898755473614,1.5938733228808726,1.27792938346661,3.9737487994847354,3.948500020160944,10.79,ANOMALY
83,2015-05-16T00:00:00.000Z,161.95507001012,523.858757906231,54.1700302272401,64.4991000716663,0.568869779827501,1.4666402049339424,4.225721111512199,4.49021867961982,10.75,ANOMALY
6,2015-01-01T06:00:00.000Z,136.878588459334,492.08841963376,149.003582009094,22.9732894818614,2.185756548271884,0.8634886119229567,4.357526570103537,3.2421877598684072,10.65,ANOMALY


probability of failure within next 24 hours

In [0]:
%sql
WITH telemetry_failure AS (

    SELECT
        t.machineID,
        t.datetime,

        CASE
            WHEN EXISTS (
                SELECT 1
                FROM tseries.tseries.silver_failures f
                WHERE f.machineID = t.machineID
                  AND f.datetime > t.datetime
                  AND f.datetime <= t.datetime + INTERVAL 24 HOURS
            )
            THEN 1
            ELSE 0
        END AS failure_next_24h

    FROM tseries.tseries.silver_telemetry t
)

SELECT
    machineID,

    COUNT(*) AS telemetry_windows,

    SUM(failure_next_24h) AS failure_windows,

    ROUND(
        100.0 * AVG(failure_next_24h),
        2
    ) AS historical_failure_rate_percent

FROM telemetry_failure

GROUP BY machineID

ORDER BY historical_failure_rate_percent DESC;

machineID,telemetry_windows,failure_windows,historical_failure_rate_percent
99,8761,453,5.17
37,8761,336,3.84
98,8761,336,3.84
20,8761,312,3.56
22,8761,309,3.53
17,8761,309,3.53
23,8761,288,3.29
71,8761,288,3.29
83,8761,285,3.25
85,8761,264,3.01


likely to failure

In [0]:
%sql
SELECT
    failure AS component,
    COUNT(*) AS failure_count,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS failure_percentage

FROM tseries.tseries.silver_failures

GROUP BY failure

ORDER BY failure_count DESC;

component,failure_count,failure_percentage
COMP2,259,34.03
COMP1,192,25.23
COMP4,179,23.52
COMP3,131,17.21


In [0]:
%sql
SELECT
    f.failure AS component,
    e.errorID,
    COUNT(*) AS related_error_events

FROM tseries.tseries.silver_failures f

INNER JOIN tseries.tseries.silver_errors e
    ON f.machineID = e.machineID
   AND e.datetime <= f.datetime
   AND e.datetime >= f.datetime - INTERVAL 48 HOURS

GROUP BY
    f.failure,
    e.errorID

ORDER BY
    component,
    related_error_events DESC;

component,errorID,related_error_events
COMP1,error1,194
COMP1,error3,15
COMP1,error4,11
COMP1,error2,11
COMP1,error5,7
COMP2,error3,266
COMP2,error2,262
COMP2,error1,19
COMP2,error4,18
COMP2,error5,14


remaining useful data

In [0]:
%sql
WITH maintenance_failure AS (

    SELECT
        m.machineID,
        m.datetime AS maintenance_time,
        m.comp,

        MIN(f.datetime) AS next_failure_time

    FROM tseries.tseries.silver_maint m

    LEFT JOIN tseries.tseries.silver_failures f
        ON m.machineID = f.machineID
       AND f.datetime > m.datetime

    GROUP BY
        m.machineID,
        m.datetime,
        m.comp
)

SELECT
    machineID,
    comp,

    COUNT(*) AS maintenance_events,

    ROUND(
        AVG(
            DATEDIFF(
                next_failure_time,
                maintenance_time
            )
        ),
        2
    ) AS avg_days_until_failure,

    MIN(
        DATEDIFF(
            next_failure_time,
            maintenance_time
        )
    ) AS minimum_days_until_failure,

    MAX(
        DATEDIFF(
            next_failure_time,
            maintenance_time
        )
    ) AS maximum_days_until_failure

FROM maintenance_failure

WHERE next_failure_time IS NOT NULL

GROUP BY
    machineID,
    comp

ORDER BY avg_days_until_failure;

machineID,comp,maintenance_events,avg_days_until_failure,minimum_days_until_failure,maximum_days_until_failure
22,COMP4,4,16.25,15,20
51,COMP3,7,18.0,1,35
99,COMP2,11,18.18,5,45
99,COMP3,8,19.38,5,45
71,COMP3,6,19.83,15,30
23,COMP2,6,21.33,15,38
22,COMP1,10,21.5,5,45
27,COMP2,2,21.5,15,28
45,COMP1,4,23.75,5,45
71,COMP1,8,24.25,15,45


preventive maintenance events effct

In [0]:
%sql
WITH maintenance_failure AS (

    SELECT
        m.machineID,
        m.datetime AS maintenance_time,
        m.comp,

        MIN(f.datetime) AS next_failure_time

    FROM tseries.tseries.silver_maint m

    LEFT JOIN tseries.tseries.silver_failures f
        ON m.machineID = f.machineID
       AND f.datetime > m.datetime

    GROUP BY
        m.machineID,
        m.datetime,
        m.comp
)

SELECT
    comp,

    COUNT(*) AS maintenance_events,

    COUNT(next_failure_time) AS maintenance_followed_by_failure,

    ROUND(
        100.0 *
        COUNT(next_failure_time) /
        COUNT(*),
        2
    ) AS failure_after_maintenance_percent,

    ROUND(
        AVG(
            DATEDIFF(
                next_failure_time,
                maintenance_time
            )
        ),
        2
    ) AS avg_days_to_failure

FROM maintenance_failure

GROUP BY comp

ORDER BY failure_after_maintenance_percent DESC;

comp,maintenance_events,maintenance_followed_by_failure,failure_after_maintenance_percent,avg_days_to_failure
COMP4,811,701,86.44,63.13
COMP1,804,683,84.95,60.49
COMP2,863,727,84.24,60.71
COMP3,808,680,84.16,60.64


mtbf by machine model


In [0]:
%sql
WITH failure_intervals AS (

    SELECT
        f.machineID,
        f.datetime,
        f.failure,

        LAG(f.datetime) OVER (
            PARTITION BY f.machineID
            ORDER BY f.datetime
        ) AS previous_failure

    FROM tseries.tseries.silver_failures f
),

machine_mtbf AS (

    SELECT
        machineID,

        AVG(
            DATEDIFF(
                datetime,
                previous_failure
            )
        ) AS mtbf_days

    FROM failure_intervals

    WHERE previous_failure IS NOT NULL

    GROUP BY machineID
)

SELECT
    m.model,

    COUNT(DISTINCT m.machineID) AS machine_count,

    ROUND(
        AVG(mm.mtbf_days),
        2
    ) AS average_mtbf_days,

    ROUND(
        MIN(mm.mtbf_days),
        2
    ) AS minimum_mtbf_days,

    ROUND(
        MAX(mm.mtbf_days),
        2
    ) AS maximum_mtbf_days

FROM tseries.tseries.silver_machines m

INNER JOIN machine_mtbf mm
    ON m.machineID = mm.machineID

GROUP BY m.model

ORDER BY average_mtbf_days DESC;

model,machine_count,average_mtbf_days,minimum_mtbf_days,maximum_mtbf_days
model4,31,87.47,27.62,270.0
model3,34,54.0,21.0,107.0
model2,17,41.89,20.0,86.25
model1,16,31.28,19.22,49.14
